# LangGraph 高级概念：中间件和人在环

欢迎来到 LangGraph 高级概念！该笔记本建立在 LangGraph 101 的基础上，并为生产代理引入了两种强大的模式。

**您将学到什么：**
- **人在环** - 暂停代理以供人工审核和批准
- **中间件** - 修改执行中关键点的代理行为
- **工具审查** - 向敏感工具添加批准工作流程
- **动态行为** - 根据上下文调整代理响应

**先决条件：** 完成 `langgraph_101.ipynb`  
</br>
</br>

---
</br>

> **注意：** 这些模式对于安全性、合规性和用户控制至关重要的生产代理至关重要。

＃＃ 设置

让我们快速设置我们的环境。

In [ ]:
# 将项目根目录添加到 Python 路径，以便从 utils 模块导入
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 从集中式 utils 模块导入模型
# 这样可以避免 notebook 之间重复配置，并保持模型设置一致
from utils.models import model

# 替代方案：如果你想在 notebook 中直接定义模型，而不是使用集中式配置，请取消注释以下内容：
# from dotenv import load_dotenv
# load_dotenv(dotenv_path="../.env", override=True)
# from langchain.chat_models import init_chat_model
# model = init_chat_model("openai:o3-mini")

# 注意：对于其他提供商（Azure、Bedrock、Vertex AI），请更新 utils/models.py
# 有关切换 LLM 提供商的详细说明，请参阅 utils/models.py

## 第 1 部分：带有中断的人在环

＃＃＃ 问题

想象一下，您正在构建一个可以发送电子邮件或进行购买的代理。您不希望它自动执行这些操作 - 您首先需要人工批准！

**人机交互**让您：
- 暂停执行以供审查
- 批准、拒绝或编辑操作
- 为敏感操作添加安全控制

### 它是如何运作的

1. 代理遇到 `interrupt()` - 执行暂停
2.系统向人类展示信息
3. 人类提供输入（批准/拒绝/编辑）
4. 代理以 `Command(resume=...)` 恢复

### 示例 1：简单的审批工作流程

让我们从一个简单的例子开始 - 在发送电子邮件之前请求批准。

In [2]:
from langgraph.types import interrupt
from langchain_core.tools import tool


@tool
def send_email(to: str, subject: str, body: str) -> str:
    """向收件人发送电子邮件。"""

    # 暂停以供人工批准
    approval = interrupt(
        {
            "action": "send_email",
            "to": to,
            "subject": subject,
            "body": body,
            "message": "您想发送这封电子邮件吗？",
        }
    )

    if approval.get("approved"):  # 如果接受则为 true，如果拒绝则为 false
        # 在生产环境中，这里会真正发送电子邮件
        return f"邮件已发送给 {to}，主题为“{subject}”"
    return "电子邮件已被用户取消"


# 直接测试工具
print("工具创建成功！")
print(f"工具名称: {send_email.name}")
print(f"工具描述: {send_email.description}")

工具创建成功！
Tool name: send_email
Tool description: 向收件人发送电子邮件。


### 使用人机交互创建代理

现在让我们创建一个使用此工具的代理。 **记住：** 中断需要检查指针！

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

# 创建持久化检查点
checkpointer = MemorySaver()

# 使用电子邮件工具创建智能体
agent = create_agent(
    model=model,
    tools=[send_email],
    system_prompt="您是一位有用的电子邮件助理。当要求发送电子邮件时，请使用 send_email 工具。",
    checkpointer=checkpointer,  # 中断需要 checkpointer
)

### 运行直到中断

让我们运行代理并查看它暂停以等待批准：

In [4]:
from langchain.messages import HumanMessage
from langsmith import uuid7

# 为此对话创建一个唯一线程
config = {"configurable": {"thread_id": uuid7()}}

# 运行智能体并观察它暂停等待批准
result = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="发送电子邮件至 alice@example.com，主题为“明天开会”，正文为“我们下午 3 点见面”。"
            )
        ]
    },
    config=config,
)

# 检查是否遇到中断
if "__interrupt__" in result:
    print("智能体已暂停等待批准")

    interrupt_info = result["__interrupt__"][0]

    print("中断详细信息：")
    print(f"  收件人: {interrupt_info.value['to']}")
    print(f"  主题: {interrupt_info.value['subject']}")
    print(f"  正文: {interrupt_info.value['body']}")
    print(f"  消息: {interrupt_info.value['message']}")
else:
    print("智能体已完成，无中断")

代理已暂停等待批准
中断详细信息：
  To: alice@example.com
  Subject: 明天开会
  Body: 我们下午 3 点见面
  Message: 您想发送这封电子邮件吗？


### 经批准恢复

现在让我们批准电子邮件并让代理继续：

In [5]:
from langgraph.types import Command

# 批准后恢复执行
result = agent.invoke(
    Command(resume={"approved": True}),
    config=config,
)

# 打印最终回复
print("最终回复：")
print(result["messages"][-1].content)

最终回应：
电子邮件已成功发送至 **alice@example.com**，主题为 **“明天开会”**，正文内容为 **“我们下午 3 点见面”**。请留意收件箱的回复！


### 练习：尝试拒绝电子邮件

再次运行单元格，但这次通过传递 `{"approved": False}` 拒绝电子邮件：

In [6]:
# 用于拒绝示例的新线程
config_2 = {"configurable": {"thread_id": uuid7()}}

# 运行直到中断
result = agent.invoke(
    {
        "messages": [
            HumanMessage(content="发送电子邮件至 bob@example.com 并注明“您好！”")
        ]
    },
    config=config_2,
)

# 拒绝后恢复执行
result = agent.invoke(
    Command(resume={"approved": False}),  # 拒绝电子邮件
    config=config_2,
)

print("最终回复：")
print(result["messages"][-1].content)

最终回应：
看起来邮件发送被取消了。请问您希望我如何调整？例如，是否需要在邮件正文中添加更多内容，或者修改主题？请告诉我您的需求，我再为您发送。


## 第 2 部分：高级模式 - 执行前编辑

有时您想要**编辑**工具调用，而不仅仅是批准/拒绝它。让我们增强我们的工具：

In [7]:
@tool
def send_email_v2(to: str, subject: str, body: str) -> str:
    """向收件人发送电子邮件。"""

    # 暂停进行人工审核
    response = interrupt(
        {
            "action": "send_email",
            "to": to,
            "subject": subject,
            "body": body,
            "message": "查看此电子邮件。您可以批准、拒绝或编辑它。",
        }
    )

    # 处理不同的响应类型
    if response["type"] == "approve":
        return f"邮件已发送给 {to}，主题为“{subject}”"

    if response["type"] == "reject":
        return "电子邮件已取消"

    if response["type"] == "edit":
        # 使用编辑后的值
        to = response.get("to", to)
        subject = response.get("subject", subject)
        body = response.get("body", body)
        return f"""邮件已按编辑内容发送：
收件人: {to}
主题: {subject}
正文: {body}"""

    return "未知回应"


# 使用增强工具创建新智能体
agent_v2 = create_agent(
    model=model,
    tools=[send_email_v2],
    system_prompt="您是一位有用的电子邮件助理。",
    checkpointer=MemorySaver(),
)

In [8]:
# 运行并编辑电子邮件
config_3 = {"configurable": {"thread_id": uuid7()}}

# 运行直到中断
result = agent_v2.invoke(
    {
        "messages": [
            HumanMessage(content="向 team@example.com 发送有关会议的电子邮件")
        ]
    },
    config=config_3,
)

print("暂停审核...")

暂停审核...


现在让我们编辑电子邮件主题，使其成为紧急会议！

In [9]:
# 使用编辑内容恢复执行
result = agent_v2.invoke(
    Command(
        resume={
            "type": "edit",
            "subject": "紧急：今天下午 2 点开会",  # 我们已编辑电子邮件主题
            "body": "这是编辑后的电子邮件正文，包含更多详细信息。",
        }
    ),
    config=config_3,
)

print("最终回复：")
print(result["messages"][-1].content)

最终回应：
当然！在发送这封邮件之前，我想确认一些信息，以确保邮件内容准确无误：

请提供以下细节：

1. **邮件主题** — 您希望主题是什么？
2. **会议名称/类型** — 是什么会议？（例如：项目进度会、周例会、策略讨论会等）
3. **会议时间** — 具体日期和时间？
4. **会议地点/方式** — 线上（如 Zoom/腾讯会议链接）还是线下（具体会议室）？
5. **邮件内容要点** — 除了通知会议，还需要包含哪些信息？（例如：议程、准备事项、参会人员等）
6. **发件人署名** — 署名应该写谁的名字？

请告诉我以上信息，我马上为您撰写并发送！✉️


## 第 3 部分：中间件简介

**中间件**提供对代理循环的细粒度控制。它可以让您：
- 检查模型调用之前/之后的状态
- 动态修改模型请求
- 在关键执行点添加自定义逻辑

### 代理循环

 ```
Input --> [before_model] --> [wrap_model_call] --> Model --> [after_model] --> Tools --> ...
``` 

中间件挂钩到这个循环：
- **`before_model`** - 在模型执行之前运行，可以更新状态
- **`wrap_model_call`** - 包装模型调用，控制何时/如何调用模型
- **`after_model`** - 在模型执行之后、工具之前运行

### 两种钩子样式

**节点式挂钩**按顺序运行：
- `before_agent` 、 `before_model` 、 `after_model` 、 `after_agent` 
- 适合日志记录、验证、状态更新

**包裹式钩子**拦截执行：
- `wrap_model_call` , `wrap_tool_call` 
- 完全控制处理程序调用
- 适合重试、缓存、转换

### 示例1：动态系统提示

让我们创建根据用户角色更改系统提示的中间件：

In [10]:
from langchain.agents.middleware import ModelRequest, dynamic_prompt
from typing import TypedDict


# 定义上下文 schema
class Context(TypedDict):
    user_role: str


# 使用装饰器创建中间件
@dynamic_prompt
def dynamic_prompt_middleware(request: ModelRequest) -> str:
    """根据用户角色调整系统提示。"""

    user_role = request.runtime.context.get("user_role", "general")

    if user_role == "expert":
        return "你是面向专家的 AI 助手。请提供详细的技术回答和代码示例。"
    if user_role == "beginner":
        return "你是面向初学者的 AI 助手。请用简单语言解释概念，避免术语。"
    return "你是一个有用的 AI 助手。"

In [11]:
from langchain_core.tools import tool


@tool
def explain_concept(concept: str) -> str:
    """解释一个编程概念。"""
    explanations = {
        "async": "异步编程允许代码无阻塞地运行。",
        "recursion": "递归是指函数调用自身。",
    }
    return explanations.get(concept.lower(), "未找到概念。")


# 使用中间件创建智能体
agent_with_middleware = create_agent(
    model=model,
    tools=[explain_concept],
    middleware=[dynamic_prompt_middleware],
    context_schema=Context,
)

### 测试不同的用户角色

让我们看看代理如何根据用户角色做出不同的响应：

In [ ]:
# 专家用户
print("=" * 50)
print("专家用户")
print("=" * 50)

result = agent_with_middleware.invoke(
    {"messages": [HumanMessage(content="解释异步编程")]},
    context={"user_role": "expert"},
)
print(result["messages"][-1].content)
print()

# 初学者用户
print("=" * 50)
print("初学者用户")
print("=" * 50)

result = agent_with_middleware.invoke(
    {"messages": [HumanMessage(content="解释异步编程")]},
    context={"user_role": "beginner"},
)
print(result["messages"][-1].content)

专家用户
## 异步编程详解

异步编程是一种编程范式，允许程序在等待某个操作完成（如I/O操作、网络请求、文件读写）的同时，继续执行其他任务，而不是阻塞等待。这可以显著提高程序的效率和资源利用率。

### 核心概念

#### 1. 同步 vs 异步

```
同步（Synchronous）：
  任务A → 等待 → 任务A完成 → 任务B → 等待 → 任务B完成

异步（Asynchronous）：
  任务A开始 → 任务A继续（后台执行）→ ...
  同时任务B开始 → 任务B继续 → ...
  当A完成时 → 处理A的结果
```

#### 2. 关键术语

- **阻塞（Blocking）**：调用方等待操作完成期间无法做其他事
- **非阻塞（Non-blocking）**：调用方发起操作后立即返回，不等待结果
- **事件循环（Event Loop）**：持续检查并处理异步任务的循环机制
- **回调（Callback）**：操作完成时被调用的函数
- **Promise/Future**：代表未来可能完成的操作结果的占位符
- **async/await**：异步编程的语法糖，让异步代码看起来像同步代码

### 各语言实现示例

#### JavaScript（最典型的异步语言）

```javascript
// 基于回调
fs.readFile('file.txt', (err, data) => {
  if (err) throw err;
  console.log(data.toString());
});

// 基于Promise
fetch('https://api.example.com/data')
  .then(response => response.json())
  .then(data => console.log(data))
  .catch(err => console.error(err));

// async/await（语法糖）
async function getData() {
  try {
    const response = await fetch('https://api.example.com/data');
    const data = await respons

### 示例 2：自定义中间件 - 请求记录器

中间件可让您连接到代理循环并查看每一步发生的情况。这对于调试和理解代理的工作原理非常有用。

**代理循环：**
用户输入 --> [before_model] --> [wrap_model_call] --> 模型 --> [after_model] --> 工具 --> ...

**我们将构建什么：**
在每一步打印信息的记录器：
- **模型之前** - 对话中有多少条消息？
- **包装模型调用** - 正在使用哪些模型和工具？
- **模型之后** - 模型是否调用了工具或给出了最终答案？

这就像添加 debug `print()` 语句，但是以一种干净、可重用的方式！

让我们创建记录模型调试请求的中间件：

In [ ]:
from langchain.agents.middleware import (
    AgentMiddleware,
    AgentState,
    ModelRequest,
    ModelResponse,
)
from typing import Any, Callable


class RequestLoggerMiddleware(AgentMiddleware):
    """记录所有模型请求，方便调试。"""

    def before_model(self, state: AgentState, runtime) -> dict[str, Any] | None:
        """在模型执行之前记录。"""
        message_count = len(state.get("messages", []))
        print(f"[模型前] 正在处理 {message_count} 条消息")
        return None  # 不修改状态

    def wrap_model_call(
        self,
        request: ModelRequest,
        handler: Callable[[ModelRequest], ModelResponse],
    ) -> ModelResponse:
        """记录模型请求详细信息并调用处理程序。"""
        print("  [模型请求]")
        print(f"   模型: {request.model if hasattr(request, 'model') else 'default'}")
        print(f"   可用工具数: {len(request.tools) if request.tools else 0}")

        # 调用实际的模型处理程序
        return handler(request)

    def after_model(self, state: AgentState, runtime) -> dict[str, Any] | None:
        """在模型执行之后记录。"""
        last_message = state["messages"][-1]
        if hasattr(last_message, "tool_calls") and last_message.tool_calls:
            print(f" [模型后] 模型请求了 {len(last_message.tool_calls)} 次工具调用")
        else:
            print(" [模型后] 模型给出了最终回复")
        return None  # 不修改状态

In [14]:
# 使用记录器中间件创建智能体
agent_with_logger = create_agent(
    model=model,
    tools=[explain_concept],
    middleware=[RequestLoggerMiddleware()],
)

### 期待什么

当我们使用记录器运行代理时，您将实时看到执行流程：

**第一次迭代：**
1. `[BEFORE MODEL]` - 显示我们开始的消息数量
2. `[MODEL REQUEST]` - 显示可用的模型和工具（来自wrap_model_call）
3. `[AFTER MODEL]` - 模型决定调用`explain_concept`工具

**第二次迭代（工具执行后）：**
1. `[BEFORE MODEL]` - 现在我们有更多消息（包括工具结果）
2. `[MODEL REQUEST]` - 再次型号信息
3. `[AFTER MODEL]` - 模型提供最终答案（无需更多工具）

这使您可以详细了解代理的决策过程。

让我们运行一下：

In [ ]:
# 运行并观察日志
print("\n" + "=" * 50)
print("使用记录器运行智能体")
print("=" * 50 + "\n")

result = agent_with_logger.invoke(
    {"messages": [{"role": "user", "content": "解释递归"}]}
)

print("\n" + "=" * 50)
print("最终回复")
print("=" * 50)
print(result["messages"][-1].content)


使用记录器运行代理

[BEFORE MODEL] Processing 1 messages
  [MODEL REQUEST]
   Model: metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10'}} profile={'name': 'DeepSeek V4 Flash', 'release_date': '2026-04-24', 'last_updated': '2026-04-24', 'open_weights': True, 'max_input_tokens': 1000000, 'max_output_tokens': 384000, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True} client=<openai.resources.chat.completions.completions.Completions object at 0x1134cacf0> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x1134cb770> root_client=<openai.OpenAI object at 0x112ffdbe0> root_async_client=<openai.AsyncOpenAI object at 0x1134cae40> model_name='deepseek-v4-flash' model_kwargs={} openai_api_key=Secre

## 第 4 部分：结合中间件和人机交互

让我们将人机交互和中间件结合起来，形成一个可投入生产的代理：

In [16]:
# 需要批准的敏感工具
@tool
def delete_database(database_name: str) -> str:
    """删除数据库。这很危险！"""

    response = interrupt(
        {
            "action": "delete_database",
            "database_name": database_name,
            "warning": "这将永久删除数据库！",
            "message": "你绝对确定吗？",
        }
    )

    if response.get("confirmed"):
        return f"数据库 {database_name!r} 已被删除（模拟）"
    return "数据库删除已取消"


# 跟踪危险操作的中间件
class SafetyMiddleware(AgentMiddleware):
    """添加安全检查和日志记录。"""

    name = "safety_checker"

    def after_model(self, state: AgentState, runtime) -> dict[str, Any] | None:
        """检查危险的工具调用。"""
        last_message = state["messages"][-1]

        if hasattr(last_message, "tool_calls") and last_message.tool_calls:
            for tool_call in last_message.tool_calls:
                if "delete" in tool_call["name"].lower():
                    print("[安全] 检测到危险操作！")
                    print(f"   工具: {tool_call['name']}")
                    print(f"   参数: {tool_call['args']}")

        return None


# 创建生产智能体
production_agent = create_agent(
    model=model,
    tools=[delete_database],
    middleware=[SafetyMiddleware()],
    checkpointer=MemorySaver(),
)

### 期待什么：行动中的分层安全

当我们尝试危险操作时，您会看到**两个**安全机制都会激活：

**第 1 层 - 中间件检测：**
  - `[SAFETY] Dangerous operation detected!` - 中间件发现删除操作
  - 记录审计跟踪的工具名称和参数

**第 2 层 - 人工批准（中断）：**
  - 代理执行在 `interrupt()` 处暂停 
  - 向人工审阅者显示警告消息
  - 在明确批准之前不会继续执行

**这是深度防御：** 中间件监视所有操作，而中断则强制人类批准关键操作。

In [17]:
# 测试组合模式
config_4 = {"configurable": {"thread_id": uuid7()}}

print("\n" + "=" * 50)
print("危险操作尝试")
print("=" * 50 + "\n")

# 运行直到中断
result = production_agent.invoke(
    {"messages": [HumanMessage(content="删除 production_db 数据库")]},
    config=config_4,
)

if "__interrupt__" in result:
    interrupt_info = result["__interrupt__"][0]
    print("需要人工批准：")
    print(f"   {interrupt_info.value['warning']}")
    print(f"   数据库: {interrupt_info.value['database_name']}")

print("（在真实应用中，人类会在继续之前检查这一点）")


危险的操作尝试

[安全] 检测到危险操作！
   Tool: delete_database
   Args: {'database_name': 'Production_db'}
需要人工批准：
   这将永久删除数据库！
   Database: Production_db
（在真实的应用程序中，人们会在继续之前检查这一点）


## 要点

### 人在环（中断）
- 使用 `interrupt()` 暂停执行
- 需要 `checkpointer` 才能持久
- 继续使用 `Command(resume=value)` 
- 非常适合审批工作流程和敏感操作

### 中间件
- **节点式挂钩**： `before_model` 、 `after_model` - 顺序逻辑、验证、日志记录
- **包裹式挂钩**： `wrap_model_call` 、 `wrap_tool_call` - 完全控制、重试、转换
- **装饰器**： `@dynamic_prompt` 、 `@before_model` 、 `@wrap_model_call` 用于快速中间件
- **类**：复杂、可重用组件的子类 `AgentMiddleware`

### 何时使用什么？

**在以下情况下使用中断：**
- 你的行动需要人类的批准
- 您想要查看/编辑工具调用
- 您需要验证用户输入

**在以下情况下使用中间件：**
- 您需要动态修改代理行为
- 您想要添加日志记录/监控
- 您需要执行政策（代币限制、安全检查）
- 您想要根据上下文个性化响应

**节点式与包裹式：**
- 用于顺序操作的节点样式（日志记录、验证）
- 控制流的环绕式（重试、回退、缓存）

## 练习（可选）

尝试构建一个代理：
1.有购买工具
2.使用中间件检查购买金额是否超过1000美元
3. 如果超过 1000 美元，则使用中断请求批准
4. 如果低于 1000 美元，则自动处理

提示：将 `before_model` 中间件与条件 `interrupt()` 逻辑相结合！

In [ ]:
# 你的代码在这里！
# 挑战：构建采购审批智能体

# @tool
# def make_purchase(item: str, amount: float) -> str:
#     ...

# class PurchaseLimitMiddleware(AgentMiddleware):
#     ...

## 后续步骤

您现在拥有用于构建生产代理的强大工具！

**继续你的旅程：**
1. 查看多代理系统的 `multi_agent.ipynb`
2.探索内置中间件（Summarization、Anthropic Prompt Caching）
3. 为您的用例构建您自己的自定义中间件
4.添加LangSmith用于调试和监控

**资源：**
- [中间件文档]( https://docs.langchain.com/oss/python/langchain/middleware) 
- [人在环指南]( https://docs.langchain.com/oss/python/langchain/human-in-the-loop) 
- [LangGraph 文档]( https://langchain-ai.github.io/langgraph/)

</br>
</br>
